In [1]:
import torch
import numpy as np

from dinosaw.utils import do_2D_pca, add_custom_font, get_features
from dinosaw.wrappers import get_model, get_models, ModelTypes, MODEL_NAMES, PretrainedViTWrapper
# from dinosaw.models.vit_wrapper import PretrainedViTWrapper, MODEL_LIST
import dinosaw.utils as utils
from skimage.transform import resize

from os import listdir
from PIL import Image

import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

from typing import Literal, TypeAlias

SEED = 100001
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = 'cuda:0'

/home/ronan/dino-saw/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
selected_models: tuple[ModelTypes, ...] = ('dinov2_s', 'dvt_dinov2_s', 'alibi_coco_dinov2_s')

models = get_models(selected_models, DEVICE, True, '../../models/checkpoints',  conf_dir='../../models/dinov3')
S = models[selected_models[0]].stride
titles = [MODEL_NAMES[m] for m in selected_models]
n_dims = models[selected_models[0]].embed_dim

2026-07-24 08:18:27 | I | factory.py                 : 152 | Building wrapper 'dinov2_s' on device cuda:0
2026-07-24 08:18:27 | I | factory.py                 : 131 | Building backbone with config: BackboneConfig(backbone_type='timm', model_arch='dinov2_s', pretrained=True, checkpoint_path=None, model_conf_path='../../models/dinov3', stride=None, remove_pos_embed=False, add_flash_attn=False, dynamic_img_size=True, dynamic_img_pad=False, modifications=[], dtype=torch.float32)


2026-07-24 08:18:27 | I | wrapper.py                 :  48 | Initialized PretrainedViTWrapper - name: '', arch: 'None', device: cuda:0, stride: (14, 14), patch_size: (14, 14), embed_dim: 384, num_blocks: 12
2026-07-24 08:18:27 | I | factory.py                 : 152 | Building wrapper 'dvt_dinov2_s' on device cuda:0
2026-07-24 08:18:27 | I | factory.py                 : 131 | Building backbone with config: BackboneConfig(backbone_type='timm', model_arch='dinov2_s', pretrained=True, checkpoint_path=None, model_conf_path='../../models/dinov3', stride=None, remove_pos_embed=False, add_flash_attn=False, dynamic_img_size=True, dynamic_img_pad=False, modifications=[], dtype=torch.float32)
2026-07-24 08:18:27 | I | wrapper.py                 :  48 | Initialized PretrainedViTWrapper - name: '', arch: 'None', device: cuda:0, stride: (14, 14), patch_size: (14, 14), embed_dim: 384, num_blocks: 12
2026-07-24 08:18:27 | I | factory.py                 : 152 | Building wrapper 'alibi_coco_dinov2_s' on

In [3]:
image_names = ["diff_shapes_518.png", "coral.png", "dog.png", "nmc_zoom.png"]
embeddings = []

for img_fname in image_names:
    for _, model in models.items():
        _img = Image.open(f"data/3D_plot/{img_fname}").convert("RGB")
                    
        _img = _img.resize((518, 518))

        emb_np = get_features(model, _img,)
        
        pca_emb = do_2D_pca(emb_np, n_components=3, post_norm="minmax")
        embeddings.append(pca_emb)

2026-07-24 08:18:28 | I | wrapper.py                 :  92 | Processing image, size: [518, 518]
2026-07-24 08:18:28 | I | wrapper.py                 : 192 | Forward Features: x: [1,3,518,518] -> f: [1,384,37,37]
2026-07-24 08:18:28 | I | wrapper.py                 :  92 | Processing image, size: [518, 518]
2026-07-24 08:18:28 | I | wrapper.py                 : 192 | Forward Features: x: [1,3,518,518] -> f: [1,384,37,37]
2026-07-24 08:18:28 | I | wrapper.py                 :  92 | Processing image, size: [518, 518]
2026-07-24 08:18:28 | I | wrapper.py                 : 192 | Forward Features: x: [1,3,518,518] -> f: [1,384,37,37]
2026-07-24 08:18:28 | I | wrapper.py                 :  92 | Processing image, size: [518, 518]
2026-07-24 08:18:28 | I | wrapper.py                 : 192 | Forward Features: x: [1,3,518,518] -> f: [1,384,37,37]
2026-07-24 08:18:28 | I | wrapper.py                 :  92 | Processing image, size: [518, 518]
2026-07-24 08:18:28 | I | wrapper.py                 : 1

In [94]:
W, H = 7.5, 2.5
def make_figure(
    n_rows: int,
    n_angles: int,
    n_spacer_rows: int = 1,
    rel_spacer_w: float = 0.2,
    n_models: int = 2,
    n_spacer_cols: int = 1,
    rel_spacer_col_w: float = 0.08,
) -> tuple[plt.Figure, GridSpec]:
    n_embed_cols = 1 + n_angles
    n_panel_cols = 1 + n_spacer_cols + n_embed_cols  # image + spacer + embedding columns
    n_cols = 2 * n_panel_cols

    rel_heights = [1 for _ in range(n_rows + n_spacer_rows)]
    for i in range(n_spacer_rows):
        rel_heights[n_models * (i + 1) + i // n_models] = rel_spacer_w

    width_ratios = [1.0 for _ in range(n_cols)]
    for panel_idx in range(2):
        spacer_col = panel_idx * n_panel_cols + 1
        for i in range(n_spacer_cols):
            width_ratios[spacer_col + i] = rel_spacer_col_w

    fig = plt.figure(figsize=(W, 2 * 2.3))
    gs = GridSpec(
        n_rows + n_spacer_rows,
        n_cols,
        figure=fig,
        width_ratios=width_ratios,
    )

    return fig, gs


def panel_col_start(panel_idx: int, n_angles: int, n_spacer_cols: int = 1) -> int:
    n_embed_cols = 1 + n_angles
    n_panel_cols = 1 + n_spacer_cols + n_embed_cols
    return panel_idx * n_panel_cols


def embedding_col_start(panel_idx: int, n_angles: int, n_spacer_cols: int = 1) -> int:
    return panel_col_start(panel_idx, n_angles, n_spacer_cols) + 1 + n_spacer_cols


def get_axes_row(fig, gs, row_idx: int, col_idx: int = 0, n_angles: int = 5, n_spacer_rows: int = 1, n_models: int = 2) -> list[plt.Axes]:
    axs = []
    col = 0

    offset = (row_idx // n_models) * n_spacer_rows
    row_idx += offset

    # One 2D axis for PCA image, followed by n_angles 3D axes.
    ax = fig.add_subplot(gs[row_idx, col_idx + col])
    axs.append(ax)
    col += 1

    for _ in range(n_angles):
        ax = fig.add_subplot(gs[row_idx, col_idx + col], projection='3d')
        axs.append(ax)
        col += 1

    return axs

def hide_axes(ax: plt.Axes):
    ax.set_xticks([])
    ax.set_yticks([])
    if hasattr(ax, 'set_zticks'):
        ax.set_zticks([])

def plot_row_of_embeddings(emb_2D: np.ndarray, axs: list[plt.Axes], n_angles: int=5, decimation_rate: int=1):

    axs[0].imshow(emb_2D)
    hide_axes(axs[0])

    for i in range(1, n_angles + 1):
        ax = axs[i]

        j = i - 1
        angle_deg = 270 + j * (180 // n_angles)

        ax.view_init(azim=angle_deg)

        embed_flat = emb_2D.reshape(-1, 3)

        xs = embed_flat[:, 0] - 0.5
        ys = embed_flat[:, 1] - 0.5
        zs = embed_flat[:, 2] - 0.5

        embed_flat = embed_flat[::decimation_rate]

        xs = xs[::decimation_rate]
        ys = ys[::decimation_rate]
        zs = zs[::decimation_rate]

        colours = embed_flat
        ax.scatter(xs, ys, zs, marker='o', c=colours, s=0.2, alpha=1, zorder=-1, rasterized=True)
        ax.set_rasterization_zorder(0)

        hide_axes(ax)
    return 

In [106]:

plt.style.use("thesis.mplstyle")
plt.rcParams['text.usetex'] = False
add_custom_font('resources/fonts', 'Grotesk')
n_rows = (len(image_names) // 2) * len(models)  # 2 models per image
n_angles = 3

n_spacer_rows = 0
n_spacer_cols = 1

spacer_w = 0.225

fig, gs = make_figure(
    n_rows=n_rows,
    n_angles=n_angles,
    n_spacer_rows=n_spacer_rows,
    rel_spacer_w=spacer_w,
    n_models=len(models),
    n_spacer_cols=n_spacer_cols,
    rel_spacer_col_w=spacer_w,
 )
print(gs)

j = 0
for i, img_fname in enumerate(image_names):
    _img = Image.open(f"data/3D_plot/{img_fname}").convert("RGB")
    _img = _img.resize((518, 518))

    col_idx = embedding_col_start(i // 2, n_angles=n_angles, n_spacer_cols=n_spacer_cols)

    for title, model in zip(titles, models):
        emb = embeddings[j]
        axs = get_axes_row(fig, gs, j % 6, col_idx, n_angles=n_angles, n_spacer_rows=n_spacer_rows, n_models=len(models))
        weight = 500
        title = title.replace("(COCO)", "")
        if 'alibi' in title.lower():
            title = "ALiBi-Dv2"
            weight = 700
        axs[0].set_ylabel(f"{title}", fontweight=weight,)
        plot_row_of_embeddings(emb, axs, n_angles=n_angles,)
        j += 1


N = len(models)
for i, img_fname in enumerate(image_names):
    _img = Image.open(f"data/3D_plot/{img_fname}").convert("RGB")
    _img = _img.resize((518, 518))

    offset = (i % 2) * n_spacer_rows
    img_col = panel_col_start(i // 2, n_angles=n_angles, n_spacer_cols=n_spacer_cols)
    img_ax = fig.add_subplot(gs[(i % 2) * N + offset : (1 + i % 2) * (N) + offset, img_col])
    img_ax.imshow(_img)
    hide_axes(img_ax)


# plt.tight_layout()
SAVE = True
if SAVE:
    plt.savefig("saved/05.pdf", dpi=300, bbox_inches='tight', pad_inches=0.05)
    plt.close()


findfont: Failed to find font weight normal, now using 300.


GridSpec(6, 12, width_ratios=[1.0, 0.225, 1.0, 1.0, 1.0, 1.0, 1.0, 0.225, 1.0, 1.0, 1.0, 1.0])


findfont: Failed to find font weight 500, now using 300.
